In [3]:
import pandas as pd 
import altair as alt

alt.data_transformers.disable_max_rows()

url = "https://github.com/UIUC-iSchool-DataViz/is445_data/raw/main/ufo-scrubbed-geocoded-time-standardized-00.csv"

cols = ["datetime", "city", "state", "country", "shape",
        "duration_seconds", "duration_text", "comments",
        "date_posted", "latitude", "longitude"]

ufo_df = pd.read_csv(url, header=None, names=cols, on_bad_lines="skip")
ufo_df.head()

,datetime,city,state,country,shape,duration_seconds,duration_text,comments,date_posted,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700.0,45 minutes,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.384210,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.200000,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611


In [6]:
ufo_df["datetime"] = pd.to_datetime(ufo_df["datetime"], errors="coerce")
ufo_df["year"] = ufo_df["datetime"].dt.year
ufo_df["latitude"] = pd.to_numeric(ufo_df["latitude"], errors="coerce")
ufo_df["longitude"] = pd.to_numeric(ufo_df["longitude"], errors="coerce")

ufo_df = ufo_df.dropna(subset=["datetime", "latitude", "longitude", "shape"])
ufo_df = ufo_df[ufo_df["year"] >= 1990]

top_shapes = ufo_df["shape"].value_counts().head(6).index.tolist()
ufo_top = ufo_df[ufo_df["shape"].isin(top_shapes)].copy()

print(top_shapes, len(ufo_top))


['light', 'triangle', 'circle', 'fireball', 'other', 'unknown'] 46583


In [7]:
yearly = ufo_top.groupby(["year", "shape"]).size().reset_index(name="count")
yearly.head()


,year,shape,count
0,1990,circle,15
1,1990,fireball,10
2,1990,light,40
3,1990,other,34
4,1990,triangle,37


In [10]:
plot_one = alt.Chart(yearly).mark_line(point=True).encode(
    x=alt.X("year:Q", title="Year", axis=alt.Axis(format="d")),
    y=alt.Y("count:Q", title="# of sightings"),
    color=alt.Color("shape:N", title="Shape",
                    scale=alt.Scale(scheme="tableau10")),
    tooltip=["year:Q", "shape:N", "count:Q"]
).properties(
    width=500,
    height=400,
    title="UFO Sightings per Year by Shape 1990 and Onwards"
)
plot_one

alt.Chart(...)

In [11]:
plot_one.save("plot_one.html")

In [15]:
us_only_data = ufo_top[
    (ufo_top["country"] == "us") &
    ufo_top["latitude"].between(24, 50) &
    ufo_top["longitude"].between(-125, -66)
]

shape_counts = us_only_data["shape"].value_counts().reset_index()
shape_counts.columns = ["shape", "count"]

us_sample = us_only_data.sample(n=min(15000, len(us_only_data)), random_state=42)[
    ["shape", "latitude", "longitude", "year", "city", "state"]
]

len(us_only_data), len(us_sample)

(37708, 15000)

In [18]:
color_scale = alt.Scale(domain=top_shapes, scheme="tableau10")

select_shape = alt.selection_point(fields=["shape"])

bars = alt.Chart(shape_counts).mark_bar().encode(
    y=alt.Y("shape:N", sort="-x", title="Shape"),
    x=alt.X("count:Q", title="# of sightings"),
    color=alt.condition(
        select_shape,
        alt.Color("shape:N", scale=color_scale, legend=None),
        alt.value("lightgray")
    ),
    tooltip=["shape:N", "count:Q"]
).add_params(select_shape).properties(
    width=250, height=350,
    title="Click a shape to filter the map"
)

us_states = alt.topo_feature(
    "https://cdn.jsdelivr.net/npm/vega-datasets@v1.29.0/data/us-10m.json",
    "states"
)

background = alt.Chart(us_states).mark_geoshape(
    fill="lightgray", stroke="white"
).project("albersUsa")

points = alt.Chart(us_sample).mark_circle(size=12, opacity=0.6).encode(
    longitude="longitude:Q",
    latitude="latitude:Q",
    color=alt.Color("shape:N", scale=color_scale, legend=None),
    tooltip=["city:N", "state:N", "shape:N", "year:Q"]
).transform_filter(select_shape).project("albersUsa")

map_chart = (background + points).properties(
    width=550, height=350,
    title="US Sightings sample of 15,000"
)

plot_two = alt.hconcat(bars, map_chart)
plot_two

alt.HConcatChart(...)

In [19]:
plot_two.save("plot_two.html")